# Testing the EgoTimeQA dataset

In [6]:
import json
from pathlib import Path

# Configuration
BASE_PATH = "/Users/marcushamelink/Developer/ml/IRIS-personal-tests/data"
ANNOTATIONS = {
    "egotime": BASE_PATH + "/EgoTimeQA/unified/annotations.EgoTimeQA.json",
    "ego4d": BASE_PATH + "/ego4d-data/ego4d.json",
}


In [ ]:
import json


def inspect_dataset(file_path, name):
    print(f"--- Inspecting {name} ---")
    try:
        with open(file_path) as f:
            data = json.load(f)

        # Check if it's a list or dictionary
        if isinstance(data, list):
            print(f"Root type: List with {len(data)} items")
            sample = data[0]
        elif isinstance(data, dict):
            print(f"Root keys: {list(data.keys())}")
            # Usually data is nested under a 'videos' or 'annotations' key
            if "videos" in data:
                print(f"Found 'videos' key with {len(data['videos'])} items")
                sample = data["videos"][0]
            else:
                # Just grab the first key's value to peek
                first_key = next(iter(data.keys()))
                sample = data[first_key]

        print(
            f"Sample Structure: {json.dumps(sample, indent=2, default=str)[:1000]}..."
        )  # Print first 1000 chars
    except Exception as e:
        print(f"Error reading {name}: {e}")
    print("\n")


# Inspect both datasets
inspect_dataset(ANNOTATIONS["egotime"], "EgoTimeQA (Training)")
# inspect_dataset(ANNOTATIONS["QaEgo4D_val"], "QAEgo4D (Validation)")

--- Inspecting EgoTimeQA (Training) ---
Root type: List with 302910 items
Sample Structure: {
  "video_id": "16fc2d81-e55c-4a3b-a298-b9d0272e9815",
  "sample_id": "16fc2d81-e55c-4a3b-a298-b9d0272e9815",
  "answer": "a paper",
  "question": "What did Person A pick up?",
  "moment_start_frame": 439.55793648760795,
  "moment_end_frame": 552.4420635123922,
  "wrong_answers": [
    "a pen",
    "a book",
    "a phone"
  ]
}...


--- Inspecting QAEgo4D (Validation) ---
Root type: List with 1911 items
Sample Structure: {
  "video_uid": "38737402-19bd-4689-9e74-3af391b15feb",
  "video_start_sec": 1055.0,
  "video_end_sec": 1061.0,
  "video_id": "93231c7e-1cf4-4a20-b1f8-9cc9428915b2",
  "sample_id": "ca7e11a2-cd1e-40dd-9d2f-ea810ab6a99b_0",
  "answer": "dirt from the vacuum cleaner",
  "question": "what did I put in the black dustbin?",
  "moment_start_frame": 12750.0,
  "moment_end_frame": 12930.0,
  "wrong_answers": [
    "last night's leftovers",
    "old newspapers",
    "broken toys"
  ]
}

In [15]:
"""Test script for exploring dataset JSON structure."""
import sys
from io import StringIO


def explore_json_structure(
    file_path: str, max_depth: int = 3, name: str = "Dataset"
) -> None:
    """Explore and print JSON structure recursively."""
    print(f"\n{'=' * 60}")
    print(f"Exploring: {name}")
    print(f"Path: {file_path}")
    print(f"{'=' * 60}\n")

    # Check if file exists
    if not Path(file_path).exists():
        print(f"❌ File not found: {file_path}")
        return

    try:
        with open(file_path, "r") as f:
            data = json.load(f)

        print(f"✓ File loaded successfully")
        print(f"Root type: {type(data).__name__}")

        if isinstance(data, dict):
            print(f"Number of root keys: {len(data)}")
            print(f"\nRoot keys: {list(data.keys())}\n")

            # Explore each top-level key
            for key in data.keys():
                value = data[key]
                print(f"Key: '{key}'")
                print(f"  Type: {type(value).__name__}")

                if isinstance(value, (list, dict)):
                    if isinstance(value, list):
                        print(f"  Length: {len(value)}")
                        if len(value) > 0:
                            print(f"  First item type: {type(value[0]).__name__}")
                            print(
                                f"  First item preview: {json.dumps(value[0], indent=2, default=str)[:500]}"
                            )
                    elif isinstance(value, dict):
                        print(f"  Number of keys: {len(value)}")
                        print(f"  Keys: {list(value.keys())[:10]}")  # First 10 keys
                print()

        elif isinstance(data, list):
            print(f"Number of items: {len(data)}")
            if len(data) > 0:
                print(f"First item type: {type(data[0]).__name__}")
                print(f"\nFirst item structure:")
                print(json.dumps(data[0], indent=2, default=str)[:1000])

    except json.JSONDecodeError as e:
        print(f"❌ JSON decode error: {e}")
    except Exception as e:
        print(f"❌ Error: {e}")


def get_nested_keys(
    data: dict, prefix: str = "", max_depth: int = 3, current_depth: int = 0
) -> list:
    """Recursively get all nested keys in a dictionary."""
    if current_depth >= max_depth:
        return []

    keys = []
    for key, value in data.items():
        current_key = f"{prefix}.{key}" if prefix else key
        keys.append(current_key)

        if isinstance(value, dict) and current_depth < max_depth - 1:
            keys.extend(
                get_nested_keys(value, current_key, max_depth, current_depth + 1)
            )
        elif isinstance(value, list) and len(value) > 0 and isinstance(value[0], dict):
            keys.append(f"{current_key}[0]")
            if current_depth < max_depth - 1:
                keys.extend(
                    get_nested_keys(
                        value[0], f"{current_key}[0]", max_depth, current_depth + 1
                    )
                )

    return keys


def get_first_elements(file_path: str, name: str = "Dataset") -> None:
    """Get the first element/value from each high-level key."""
    print(f"\n{'=' * 60}")
    print(f"First elements from: {name}")
    print(f"{'=' * 60}\n")

    try:
        with open(file_path, "r") as f:
            data = json.load(f)

        if not isinstance(data, dict):
            if isinstance(data, list):
                print(f"Root is a list with {len(data)} items")
                if len(data) > 0:
                    print("Using first element as data to explore")
                    data = {"root_list_item": data[0]}
                else:
                    print("Empty list, nothing to explore")
                    return
            else:
                print(f"Root is not a dictionary or list, it's a {type(data).__name__}")
                return

        for key in data.keys():
            value = data[key]
            print(f"\nKey: '{key}'")
            print(f"Type: {type(value).__name__}")

            if isinstance(value, list):
                if len(value) > 0:
                    first_elem = value[0]
                    print(f"First element:")
                    print(json.dumps(first_elem, indent=2, default=str)[:1000])
                else:
                    print("Empty list")
            elif isinstance(value, dict):
                print(f"Dictionary with keys: {list(value.keys())[:10]}")
                # Get first item from dict
                if len(value) > 0:
                    first_key = next(iter(value.keys()))
                    print(f"\nFirst dict entry (key: '{first_key}'):")
                    print(json.dumps(value[first_key], indent=2, default=str)[:1000])
            else:
                print(f"Value: {str(value)[:500]}")

            print("-" * 60)

    except Exception as e:
        print(f"❌ Error: {e}")


def main() -> None:
    """Main function to explore JSON datasets."""

    # Create output file path
    output_file = Path(BASE_PATH) / "dataset_exploration_output.txt"

    # Redirect stdout to capture print statements
    original_stdout = sys.stdout
    string_buffer = StringIO()
    sys.stdout = string_buffer

    try:
        # Explore EgoTimeQA
        # explore_json_structure(ANNOTATIONS["egotime"], name="EgoTimeQA")

        # Explore Ego4D
        # explore_json_structure(ANNOTATIONS["ego4d"], name="Ego4D")

        get_first_elements(ANNOTATIONS["egotime"], name="EgoTimeQA")
        get_first_elements(ANNOTATIONS["ego4d"], name="Ego4D")

        # Deep dive into Ego4D structure
        print(f"\n{'=' * 60}")
        print("Deep dive: Ego4D nested structure")
        print(f"{'=' * 60}\n")

        try:
            with open(ANNOTATIONS["ego4d"], "r") as f:
                ego4d_data = json.load(f)

            if isinstance(ego4d_data, dict):
                nested_keys = get_nested_keys(ego4d_data, max_depth=3)
                print("Nested keys (up to 3 levels deep):")
                for key in nested_keys[:50]:  # Show first 50
                    print(f"  {key}")
                if len(nested_keys) > 50:
                    print(f"  ... and {len(nested_keys) - 50} more")

        except Exception as e:
            print(f"Error in deep dive: {e}")

    finally:
        # Restore stdout
        sys.stdout = original_stdout

        # Get captured output
        output_content = string_buffer.getvalue()

        # Write to file
        with open(output_file, "w") as f:
            f.write(output_content)

        # Also print to console
        print(output_content)
        print(f"\n✅ Output saved to: {output_file}")


if __name__ == "__main__":
    main()



First elements from: EgoTimeQA

Root is a list with 302910 items
Using first element as data to explore

Key: 'root_list_item'
Type: dict
Dictionary with keys: ['video_id', 'sample_id', 'answer', 'question', 'moment_start_frame', 'moment_end_frame', 'wrong_answers']

First dict entry (key: 'video_id'):
"16fc2d81-e55c-4a3b-a298-b9d0272e9815"
------------------------------------------------------------

First elements from: Ego4D


Key: 'date'
Type: str
Value: 231209
------------------------------------------------------------

Key: 'version'
Type: str
Value: 2.1
------------------------------------------------------------

Key: 'description'
Type: str
Value: Ego4d Video Metadata
------------------------------------------------------------

Key: 'videos'
Type: list
First element:
{
  "video_uid": "000786a7-3f9d-4fe6-bfb3-045b368f7d44",
  "duration_sec": 415.53,
  "scenarios": [
    "jobs related to construction/renovation company\n(Director of work, tiler, plumber, Electrician, Handyman

In [13]:
import json
from pathlib import Path

# 1. Configuration
BASE_PATH = "/Users/marcushamelink/Developer/ml/IRIS-personal-tests/data"
ANNOTATIONS_PATH = Path(BASE_PATH) / "EgoTimeQA/unified/annotations.EgoTimeQA.json"
EGO4D_META_PATH = Path(BASE_PATH) / "ego4d-data/ego4d.json"

# Output paths
OUTPUT_UID_FILE = Path(BASE_PATH) / "ego4d-data/egotimeqa_uids.txt"
OUTPUT_SUBSET_JSON = Path(BASE_PATH) / "ego4d-data/egotimeqa-full.json"

# 2. Load Data
print(f"Loading annotations from: {ANNOTATIONS_PATH}")
with open(ANNOTATIONS_PATH, "r") as f:
    egotime_data = json.load(f)

print(f"Loading master metadata from: {EGO4D_META_PATH}")
with open(EGO4D_META_PATH, "r") as f:
    ego4d_data = json.load(f)

# 3. Extract Unique Video IDs (The Allowlist)
# Based on your schema description, the key is 'video_id'
target_uids = set()
for item in egotime_data:
    if "video_id" in item:
        target_uids.add(item["video_id"])

print(f"Found {len(target_uids)} unique video IDs in EgoTimeQA.")

# 4. Filter Ego4D Metadata
# We copy the main structure (date, version, etc.) but replace the 'videos' list
subset_meta = ego4d_data.copy()
filtered_videos = []

found_count = 0
for video in ego4d_data.get("videos", []):
    if video["video_uid"] in target_uids:
        filtered_videos.append(video)
        found_count += 1

subset_meta["videos"] = filtered_videos

# Sanity Check
print(f"Matched {found_count} videos in the master Ego4D manifest.")
if len(target_uids) > found_count:
    print(
        f"Warning: {len(target_uids) - found_count} IDs from EgoTimeQA were not found in ego4d.json."
    )

# 5. Export Outputs

# A: The UID text file for the CLI downloader
with open(OUTPUT_UID_FILE, "w") as f:
    for uid in sorted(list(target_uids)):
        f.write(f"{uid}\n")
print(f"Saved UID list for CLI to: {OUTPUT_UID_FILE}")

# B: The rich metadata subset JSON
with open(OUTPUT_SUBSET_JSON, "w") as f:
    json.dump(subset_meta, f, indent=2)
print(f"Saved subset metadata to: {OUTPUT_SUBSET_JSON}")


Loading annotations from: /Users/marcushamelink/Developer/ml/IRIS-personal-tests/data/EgoTimeQA/unified/annotations.EgoTimeQA.json
Loading master metadata from: /Users/marcushamelink/Developer/ml/IRIS-personal-tests/data/ego4d-data/ego4d.json
Found 5389 unique video IDs in EgoTimeQA.
Matched 0 videos in the master Ego4D manifest.
Saved UID list for CLI to: /Users/marcushamelink/Developer/ml/IRIS-personal-tests/data/ego4d-data/egotimeqa_uids.txt
Saved subset metadata to: /Users/marcushamelink/Developer/ml/IRIS-personal-tests/data/ego4d-data/egotimeqa-full.json


In [17]:
import json
from pathlib import Path

# 1. Configuration
BASE_PATH = "/Users/marcushamelink/Developer/ml/IRIS-personal-tests/data"
ANNOTATIONS_PATH = Path(BASE_PATH) / "EgoTimeQA/unified/annotations.EgoTimeQA.json"
EGO4D_META_PATH = Path(BASE_PATH) / "ego4d-data/ego4d.json"
OUTPUT_UID_FILE = Path(BASE_PATH) / "ego4d-data/egotimeqa_uids.txt"
OUTPUT_SUBSET_JSON = Path(BASE_PATH) / "ego4d-data/egotimeqa-clips.json"

# 2. Load Data
print("Loading files...")
with open(ANNOTATIONS_PATH, "r") as f:
    egotime_data = json.load(f)
with open(EGO4D_META_PATH, "r") as f:
    ego4d_data = json.load(f)

# 3. Extract Target Clip IDs
# These are the IDs from the research paper's dataset
target_clip_ids = set()
for item in egotime_data:
    # The key 'video_id' in EgoTimeQA actually corresponds to 'clip_uid' in Ego4D
    if "video_id" in item:
        target_clip_ids.add(item["video_id"])

print(f"🎯 Target: Looking for {len(target_clip_ids)} unique Clip IDs.")

# 4. Map Clips to Parent Videos
# We iterate through the 'clips' list in ego4d.json to find matches
parent_video_uids = set()
found_clips_meta = []
found_count = 0

# ego4d.json has a top-level 'clips' key containing the mapping
if "clips" in ego4d_data:
    for clip in ego4d_data["clips"]:
        if clip["clip_uid"] in target_clip_ids:
            # Found a match!
            found_count += 1

            # 1. Add the PARENT video UID to the download list
            parent_video_uids.add(clip["video_uid"])

            # 2. Save the clip metadata (Crucial: contains start/end times for your embeddings)
            found_clips_meta.append(clip)

print(
    f"✅ Match Status: Found {found_count} / {len(target_clip_ids)} clips in manifest."
)
print(
    f"📹 Download List: These clips belong to {len(parent_video_uids)} unique parent videos."
)

# 5. Export Outputs

# A: The UID text file for the CLI (contains PARENT VIDEO UIDs)
# We use parent UIDs because the standard 'full_scale' download command expects them.
with open(OUTPUT_UID_FILE, "w") as f:
    for uid in sorted(list(parent_video_uids)):
        f.write(f"{uid}\n")
print(f"💾 Saved parent video UID list to: {OUTPUT_UID_FILE}")

# B: The detailed clip metadata
# Use this later to know exactly which seconds of the video to feed into Qwen2.5VL
with open(OUTPUT_SUBSET_JSON, "w") as f:
    json.dump(found_clips_meta, f, indent=2)
print(f"💾 Saved clip metadata to: {OUTPUT_SUBSET_JSON}")


Loading files...
🎯 Target: Looking for 5389 unique Clip IDs.
✅ Match Status: Found 5389 / 5389 clips in manifest.
📹 Download List: These clips belong to 1697 unique parent videos.
💾 Saved parent video UID list to: /Users/marcushamelink/Developer/ml/IRIS-personal-tests/data/ego4d-data/egotimeqa_uids.txt
💾 Saved clip metadata to: /Users/marcushamelink/Developer/ml/IRIS-personal-tests/data/ego4d-data/egotimeqa-clips.json


In [18]:
import json
from pathlib import Path

# Config
BASE_PATH = "/Users/marcushamelink/Developer/ml/IRIS-personal-tests/data"
ANNOTATIONS_PATH = Path(BASE_PATH) / "EgoTimeQA/unified/annotations.EgoTimeQA.json"
OUTPUT_CLIP_LIST = Path(BASE_PATH) / "ego4d-data/egotimeqa_clip_uids.txt"

# Load
with open(ANNOTATIONS_PATH, "r") as f:
    egotime_data = json.load(f)

# Extract strictly the IDs used in the dataset
clip_uids = set()
for item in egotime_data:
    if "video_id" in item:
        clip_uids.add(
            item["video_id"]
        )  # Remember: In this dataset, 'video_id' is actually the Clip UID

# Save for CLI
with open(OUTPUT_CLIP_LIST, "w") as f:
    for uid in sorted(list(clip_uids)):
        f.write(f"{uid}\n")

print(f"✅ Saved {len(clip_uids)} Clip UIDs to {OUTPUT_CLIP_LIST}")


✅ Saved 5389 Clip UIDs to /Users/marcushamelink/Developer/ml/IRIS-personal-tests/data/ego4d-data/egotimeqa_clip_uids.txt


In [24]:
# Quick filter for small clips only - with duration distribution analysis
import json
from collections import Counter

BASE_PATH = "/Users/marcushamelink/Developer/ml/IRIS-personal-tests/data"
input_path = Path(BASE_PATH) / "ego4d-data/egotimeqa-clips.json"
output_txt = Path(BASE_PATH) / "ego4d-data/small_clips_uids.txt"

with open(input_path) as f:
    clips = json.load(f)

# Calculate durations and sort
durations = []
for c in clips:
    duration = c["video_end_sec"] - c["video_start_sec"]
    durations.append(duration)

# Sort durations from longest to shortest
sorted_durations = sorted(durations, reverse=True)

# Show some statistics
print(f"Total clips: {len(clips)}")
print(
    f"Longest clip: {sorted_durations[0]:.1f} seconds ({sorted_durations[0] / 60:.1f} minutes)"
)
print(f"Shortest clip: {sorted_durations[-1]:.1f} seconds")
print(f"Average duration: {sum(durations) / len(durations):.1f} seconds")
print()

# Create duration ranges and count clips in each
ranges = [
    (0, 30, "0-30 sec"),
    (30, 60, "30-60 sec (1 min)"),
    (60, 120, "1-2 min"),
    (120, 300, "2-5 min"),
    (300, 600, "5-10 min"),
    (600, 1200, "10-20 min"),
    (1200, 1800, "20-30 min"),
    (1800, 3600, "30-60 min"),
    (3600, float("inf"), "60+ min"),
]

print("Duration Distribution:")
print("-" * 50)
cumulative = 0
for min_dur, max_dur, label in ranges:
    count = sum(1 for d in durations if min_dur <= d < max_dur)
    cumulative += count
    percentage = (count / len(clips)) * 100
    cumulative_pct = (cumulative / len(clips)) * 100
    print(
        f"{label:20} {count:5} clips ({percentage:5.1f}%) | Cumulative: {cumulative:5} ({cumulative_pct:5.1f}%)"
    )

# Optional: Filter with your chosen MAX_DURATION
MAX_DURATION = 60 * 20  # 20 minutes
short_clips = [
    c["clip_uid"]
    for c in clips
    if (c["video_end_sec"] - c["video_start_sec"]) < MAX_DURATION
]

print()
print(
    f"With MAX_DURATION = {MAX_DURATION / 60:.0f} min: {len(short_clips)} clips ({len(short_clips) / len(clips) * 100:.1f}%)"
)


Total clips: 5389
Longest clip: 1200.1 seconds (20.0 minutes)
Shortest clip: 179.9 seconds
Average duration: 416.4 seconds

Duration Distribution:
--------------------------------------------------
0-30 sec                 0 clips (  0.0%) | Cumulative:     0 (  0.0%)
30-60 sec (1 min)        0 clips (  0.0%) | Cumulative:     0 (  0.0%)
1-2 min                  0 clips (  0.0%) | Cumulative:     0 (  0.0%)
2-5 min                898 clips ( 16.7%) | Cumulative:   898 ( 16.7%)
5-10 min              4340 clips ( 80.5%) | Cumulative:  5238 ( 97.2%)
10-20 min              113 clips (  2.1%) | Cumulative:  5351 ( 99.3%)
20-30 min               38 clips (  0.7%) | Cumulative:  5389 (100.0%)
30-60 min                0 clips (  0.0%) | Cumulative:  5389 (100.0%)
60+ min                  0 clips (  0.0%) | Cumulative:  5389 (100.0%)

With MAX_DURATION = 20 min: 5351 clips (99.3%)
